In [ ]:
# 경로 설정 · DATA_ROOT 바로 아래에 pcb1, pcb2, pcb3, pcb4가 있어야 합니다.
from pathlib import Path

DATA_ROOT = Path("./out_data")       # 다른 위치의 예: Path(r"D:\datasets\out_data")
OUTPUT_DIR = Path("./visa_results")  # 전처리 파일, 학습 모델, 시험 결과를 저장할 폴더

# VisA PCB 결함 분류 · EfficientNet-B0

첫 셀의 상대 경로는 Jupyter 커널의 작업 폴더를 기준으로 합니다. 다른 위치에서 열었다면 절대 경로를 입력하세요. **Run All**을 선택하면 원본 분할, 타일 생성, 학습, 시험 평가를 차례로 수행합니다. 마지막 셀에 전체 및 보드별 성능이 표시됩니다. 기준 실험은 `src3/07_train_visa.py`의 VisA 단독 학습입니다.

각 보드 폴더(`pcb1`~`pcb4`)에는 다음 구조가 필요합니다.

```text
pcb1/
  Data/Images/Normal/     정상 원본
  Data/Images/Anomaly/    불량 원본
  Data/Masks/Anomaly/     불량 원본과 파일명이 같은 마스크(확장자는 달라도 됨)
  image_anno.csv          있으면 이미지·마스크 목록을 대조
```

Python 3.10 이상의 Jupyter 커널과 PyTorch 2.5 이상을 사용합니다. CUDA를 사용할 수 있으면 GPU에서 학습합니다. EfficientNet-B0의 ImageNet 가중치는 Torch 캐시에서 읽고, 첫 사용 시 torchvision이 다운로드합니다. 빈 환경이라면 커널에 필요한 패키지를 아래 명령으로 설치할 수 있습니다.

```python
%pip install "torch>=2.5,<3" "torchvision>=0.20,<1" "numpy>=1.24,<3" "pandas>=2,<4" "Pillow>=9.5,<14" "scipy>=1.10,<2" "matplotlib>=3.8,<4" "filelock>=3"
```

In [ ]:
BOARDS = ("pcb1", "pcb2", "pcb3", "pcb4")
data_root = DATA_ROOT.expanduser().resolve()
output_dir = OUTPUT_DIR.expanduser().resolve()

required = ("Data/Images/Normal", "Data/Images/Anomaly", "Data/Masks/Anomaly")
missing = [data_root / board / folder for board in BOARDS for folder in required
           if not (data_root / board / folder).is_dir()]
if missing:
    details = "\n".join(str(path) for path in missing)
    raise FileNotFoundError(f"첫 셀의 DATA_ROOT를 확인하세요. 다음 폴더가 필요합니다.\n{details}")
print(f"데이터  {data_root}\n결과    {output_dir}")

## 실험 설정

보드 종류와 정상·불량 라벨별로 원본을 70%·15%·15%로 나눕니다. 학습의 기본값은 시드 42, 배치 48, 최대 25에포크입니다. 같은 출력 폴더에서 다시 실행하면 데이터와 설정을 대조하고 완료된 학습·평가를 재사용합니다. 새 학습을 시작하려면 첫 셀의 `OUTPUT_DIR`에 새 폴더를 지정하세요.

In [ ]:
import csv
import hashlib
import json
import math
import random
import tempfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from filelock import FileLock
from IPython.display import Image as NotebookImage, display
from PIL import Image, ImageDraw
from scipy import ndimage
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

In [ ]:
CLASSES = {"0": "normal", "1": "defect"}
SPLITS = ("train", "validation", "test")
SOURCES = ("DeepPCB", "PKU", "VisA")  # 기존 metrics.csv의 집계 행 이름
EXPERIMENTS = {"VisA": ("VisA",)}
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
COORDS = ["x0", "y0", "x1", "y1"]

cfg = {
    "paths": {"VisA": str(data_root)},
    "sources": ["VisA"],
    "output_dir": str(output_dir),
    "seed": 42,
    "split_fractions": [0.7, 0.15, 0.15],
    "tiling": {"size": 640, "overlap": 128, "min_defect_coverage": 0.5},
    "training": {
        "model": "efficientnet_b0",
        "pretrained": True,
        "image_size": 224,
        "batch_size": 48,
        "epochs": 25,
        "early_stopping_patience": 7,
        "learning_rate": 0.0002,
        "optimizer": "adamw",
        "weight_decay": 0.0001,
        "augment": True,
        "preprocessing": "rgb",
        "binary_cutoff": 128,
        "imbalance": "class_weight",
        "device": "auto",
        "workers": 0,
        "cache_inputs": True,
        "amp": True,
    },
    "threshold": {"min_recall": 0.9, "max_fpr": 0.05},
}
run_dir = output_dir / "runs" / "VisA"

### 파일과 전처리 기록

원본·타일 목록을 CSV로 저장하고 각 파일의 SHA-256을 기록합니다. 학습과 평가는 이 목록을 기준으로 데이터를 읽습니다.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
                          + "\n", encoding="utf-8")


def write_rows(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def preparation_settings(cfg):
    return {k: cfg[k] for k in ("sources", "seed", "split_fractions", "tiling")}


def load_prepared(cfg):
    directory = Path(cfg["output_dir"]) / "prepared"
    if not (directory / "summary.json").is_file():
        raise FileNotFoundError(f"전처리 셀을 먼저 실행하세요. 전처리 기록 경로: {directory}")
    report = read_json(directory / "summary.json")
    if preparation_settings(cfg) != report["settings"]:
        raise ValueError("Prepared settings differ. Use the original config or a new output_dir.")
    for name, expected in report["manifest_sha256"].items():
        if sha256(directory / name) != expected:
            raise ValueError(f"Manifest changed: {directory / name}")
    originals = pd.read_csv(directory / "originals.csv", keep_default_na=False, dtype={"board": str})
    tiles = pd.read_csv(directory / "tiles.csv", keep_default_na=False)
    return directory, originals, tiles, report

## 1. 원본 이미지와 마스크 연결

보드별로 정상은 0, 불량은 1로 기록합니다. 불량 이미지마다 파일명에서 확장자를 뺀 이름으로 마스크를 하나씩 연결하고, 이미지 크기와 파일 해시를 기록합니다.

In [ ]:
@dataclass
class Original:
    image_id: str
    source: str
    board: str
    group_id: str
    path: str
    role: str
    label: int
    width: int
    height: int
    sha256: str
    annotation: str = ""
    boxes: list = field(default_factory=list)
    split: str = ""

    def row(self):
        row = asdict(self)
        row["boxes"] = json.dumps(self.boxes, separators=(",", ":"))
        return row


def images(directory):
    if not directory.is_dir():
        raise FileNotFoundError(f"Image directory missing: {directory}")
    return sorted(p for p in directory.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)


def original(root, path, source, board, group, role, label, annotation=None):
    rel = path.relative_to(root).as_posix()
    with Image.open(path) as im:
        width, height = im.size
        im.verify()
    return Original(f"{source}/{rel}", source, board, f"{source}/{group}", rel, role, label,
                    width, height, sha256(path),
                    annotation.relative_to(root).as_posix() if annotation else "")


def check_sets(actual, expected, context):
    if actual != expected:
        extra = sorted(str(v) for v in actual - expected)[:3]
        missing = sorted(str(v) for v in expected - actual)[:3]
        raise ValueError(f"{context} mismatch: extra={extra}, missing={missing}")

In [ ]:
def visa(root):
    originals = []
    for board in BOARDS:
        base = root / board / "Data"
        refs = {}
        mask_paths = images(base / "Masks" / "Anomaly")
        masks = {p.stem: p for p in mask_paths}
        if len(masks) != len(mask_paths):
            raise ValueError(f"Ambiguous mask stems: {base}")
        anomaly_stems = set()
        for folder, label in (("Normal", 0), ("Anomaly", 1)):
            files = images(base / "Images" / folder)
            if not files:
                raise ValueError(f"Empty VisA image directory: {base / 'Images' / folder}")
            for path in files:
                mask = None
                if label:
                    if path.stem not in masks or path.stem in anomaly_stems:
                        raise ValueError(f"Expected one mask per anomaly: {path}")
                    mask = masks[path.stem]
                    anomaly_stems.add(path.stem)
                group = f"{board}/{folder}/{path.stem}"
                item = original(root, path, "VisA", board, group, folder.lower(), label, mask)
                originals.append(item)
                refs[item.path] = (item.label, item.annotation)
        check_sets(set(masks), anomaly_stems, f"{board} masks/images")
        metadata = root / board / "image_anno.csv"
        if metadata.is_file():
            with metadata.open(encoding="utf-8-sig", newline="") as stream:
                rows = list(csv.DictReader(stream))
            if len(rows) != len(refs):
                raise ValueError(f"VisA metadata row count mismatch: {metadata}")
            expected = {(r["image"], int(r["label"].lower() != "normal"), r["mask"]) for r in rows}
            check_sets({(p, *v) for p, v in refs.items()}, expected, str(metadata))
    return originals

def scan_visa(root):
    records = visa(root)
    if len({item.image_id for item in records}) != len(records):
        raise ValueError("중복된 원본 이미지 ID가 있습니다.")
    return sorted(records, key=lambda item: item.image_id)

## 2. 원본 단위로 분할하고 타일에 라벨 부여

같은 원본에서 나온 타일은 모두 같은 분할에 속합니다. 640×640 창을 128픽셀씩 겹치게 이동하고 오른쪽과 아래쪽 가장자리까지 포함합니다. 불량 마스크에서 가로·세로·대각선으로 연결된 픽셀을 하나의 결함으로 봅니다.

| 타일 라벨 | 개별 결함 면적 중 타일에 포함된 비율 |
|---|---|
| 정상 `0` | 결함 픽셀과 겹치지 않음 |
| 불량 `1` | 한 결함이라도 50% 이상 포함 |
| 부분 겹침 `-1` | 결함과 겹치며 모든 결함의 포함 비율이 50% 미만 |

학습에는 라벨 0과 1을 사용합니다. 검증·시험에서 원본 점수를 낼 때는 부분 겹침 타일까지 포함합니다.

In [ ]:
def assign_splits(records, fractions, seed):
    strata = defaultdict(lambda: defaultdict(list))
    for item in records:
        key = (item.source, item.board, item.label)
        strata[key][item.group_id].append(item)
    for key, groups in sorted(strata.items()):
        order = sorted(groups)
        random.Random(f"{seed}/{key}").shuffle(order)
        total = sum(map(len, groups.values()))
        counts = [0, 0, 0]
        for i, group in enumerate(order):
            empty = [j for j in range(3) if not counts[j]]
            # Reserve a whole group for each remaining split if there are enough groups.
            choices = empty if len(order) - i <= len(empty) else range(3)
            dest = max(choices, key=lambda j: fractions[j] * total - counts[j])
            for item in groups[group]:
                item.split = SPLITS[dest]
            counts[dest] += len(groups[group])


def starts(length, size, overlap):
    last = max(0, length - size)
    values = list(range(0, last + 1, size - overlap))
    if values[-1] != last:
        values.append(last)
    return values


def windows(width, height, tiling):
    size, overlap = tiling["size"], tiling["overlap"]
    for y in starts(height, size, overlap):
        for x in starts(width, size, overlap):
            yield (x, y, min(x + size, width), min(y + size, height))

In [ ]:
def read_mask(path, size):
    with Image.open(path) as im:
        if im.size != size:
            raise ValueError(f"Mask size {im.size} differs from image {size}: {path}")
        raw = np.asarray(im)
        mask = np.any(raw[:, :, :3] != 0, axis=2) if raw.ndim == 3 else raw != 0
    if not mask.any():
        raise ValueError(f"Empty mask for image marked Anomaly: {path}")
    return mask


@dataclass
class Region:
    bounds: tuple
    area: int
    pixels: np.ndarray | None = None


def regions(item, root):
    if not item.label:
        return []
    mask = read_mask(Path(root) / item.annotation, (item.width, item.height))
    ids, _ = ndimage.label(mask, structure=np.ones((3, 3), dtype=bool))
    components = []
    for index, (ys, xs) in enumerate(ndimage.find_objects(ids), start=1):
        pixels = ids[ys, xs] == index
        components.append(Region((xs.start, ys.start, xs.stop, ys.stop), int(pixels.sum()), pixels))
    return components

In [ ]:
def tile_target(window, defects, minimum):
    """Store the intersection and denominator of the best-covered individual defect."""
    x0, y0, x1, y1 = window
    inside = area = 0
    for defect in defects:
        left, top, right, bottom = defect.bounds
        x, y, r, b = max(x0, left), max(y0, top), min(x1, right), min(y1, bottom)
        if r <= x or b <= y:
            continue
        count = (r - x) * (b - y)
        if defect.pixels is not None:
            count = int(defect.pixels[y-top:b-top, x-left:r-left].sum())
        if count and (not area or count * area > inside * defect.area):
            inside, area = count, defect.area
    coverage = inside / area if area else 0.0
    label = 0 if inside == 0 else (1 if coverage >= minimum else -1)
    return label, coverage, inside, area


def enumerate_tiles(originals, cfg):
    output = []
    for item in originals:
        defects = regions(item, cfg["paths"][item.source])
        for box in windows(item.width, item.height, cfg["tiling"]):
            label, fraction, inside, area = tile_target(box, defects, cfg["tiling"]["min_defect_coverage"])
            output.append(dict(tile_id=len(output), image_id=item.image_id, split=item.split,
                               x0=box[0], y0=box[1], x1=box[2], y1=box[3], label=label,
                               overlap_fraction=fraction, overlap_px=inside, region_px=area))
    return output

## 3. 분할 검사와 전처리 결과

원본 ID, 그룹 ID, 파일 내용이 여러 분할에 걸치는지 검사합니다. 타일 좌표가 중복되거나 타일이 원본과 다른 분할에 있으면 전처리를 중단합니다.

In [ ]:
def audit(originals, tiles):
    tests = {}
    for key in ("image_id", "group_id", "sha256"):
        tests[f"{key}_in_multiple_splits"] = int((originals.groupby(key).split.nunique() > 1).sum())
    tests["same_bytes_conflicting_labels"] = int((originals.groupby("sha256").label.nunique() > 1).sum())
    tests["duplicate_original_id"] = int(originals.image_id.duplicated().sum())
    tests["duplicate_tile_id"] = int(tiles.tile_id.duplicated().sum())
    tests["repeated_tile_in_original"] = int(tiles.duplicated(["image_id", "x0", "y0", "x1", "y1"]).sum())
    parent_split = originals.set_index("image_id").split
    tests["tile_parent_split_mismatch"] = int((tiles.image_id.map(parent_split) != tiles.split).sum())
    tests["originals_without_tiles"] = len(set(originals.image_id) - set(tiles.image_id))
    if any(tests.values()):
        raise ValueError(f"Split or identity audit failed: {tests}")
    return dict(passed=True, violations=tests,
                repeated_image_files_within_split=int(originals.sha256.duplicated().sum()),
                augmentation="on-the-fly, training loader only")


def population(originals, tiles):
    rows = []
    sources = originals.set_index("image_id").source
    local_tiles = tiles.assign(source=tiles.image_id.map(sources))
    for source in SOURCES:
        if source not in set(originals.source):
            continue
        for split in SPLITS:
            items = originals[(originals.source == source) & (originals.split == split)]
            t = local_tiles[(local_tiles.source == source) & (local_tiles.split == split)]
            kept = set(t.loc[t.label >= 0, "image_id"])
            positive = set(t.loc[t.label == 1, "image_id"])
            rows.append(dict(source=source, split=split, groups=items.group_id.nunique(),
                             originals=len(items), original_normal=int((items.label == 0).sum()),
                             original_defect=int(items.label.sum()), all_tiles=len(t),
                             tile_normal=int((t.label == 0).sum()), tile_defect=int((t.label == 1).sum()),
                             tile_excluded=int((t.label == -1).sum()), kept_tiles=int((t.label >= 0).sum()),
                             originals_with_kept_tiles=len(kept),
                             defect_originals_without_positive_tile=int(items.label.sum()) - len(positive)))
    return rows

### 학습 타일 미리보기

학습 데이터에서 정상, 불량, 부분 겹침 타일을 한 개씩 골라 원본에서의 위치, 실제 입력, 마스크와의 겹침을 나란히 표시합니다.

In [ ]:
def annotated(image, row, root):
    overlay = image.copy()
    pen = ImageDraw.Draw(overlay)
    stroke = max(3, image.width // 350)
    for x, y, r, b in json.loads(row.boxes):
        pen.rectangle((x, y, r - 1, b - 1), outline="#e00056", width=stroke)
    if row.source == "VisA" and row.label:
        mask = read_mask(root / row.annotation, image.size)
        alpha = Image.fromarray(mask.astype("uint8") * 165)
        overlay.paste(Image.new("RGB", image.size, "#ff0066"), mask=alpha)
    return overlay


def draw_previews(directory, originals, tiles, cfg):
    directory.mkdir()
    by_id = originals.set_index("image_id")
    joined = tiles[tiles.split == "train"].merge(
        originals[["image_id", "source", "label"]].rename(columns={"label": "image_label"}), on="image_id")
    examples = []
    for source, rows in joined.groupby("source", sort=False):
        choices = []
        for label, name, image_label in [(0, "normal", 0), (1, "defect", 1), (-1, "excluded", 1)]:
            subset = rows[(rows.label == label) & (rows.image_label == image_label)]
            if not subset.empty:
                choices.append((name, subset.iloc[0]))
        if not choices:
            continue
        fig, axes = plt.subplots(len(choices), 3, squeeze=False, figsize=(10, 3 * len(choices)))
        for (name, tile), axes_row in zip(choices, axes):
            rec = by_id.loc[tile.image_id]
            root = Path(cfg["paths"][source])
            with Image.open(root / rec.path) as raw:
                image = raw.convert("RGB")
            overlay = annotated(image, rec, root)
            box = tuple(int(tile[k]) for k in ("x0", "y0", "x1", "y1"))
            crop = overlay.crop(box)
            ImageDraw.Draw(overlay).rectangle((box[0], box[1], box[2]-1, box[3]-1),
                                               outline="#00b6d9", width=max(4, image.width // 200))
            views = ((overlay, f"{source} / {rec.board}  |  tile window"),
                     (image.crop(box), f"Input tile  |  {name}"),
                     (crop, f"GT  |  coverage {tile.overlap_fraction:.3f}"))
            for ax, (view, title) in zip(axes_row, views):
                view.thumbnail((640, 640))
                ax.imshow(view)
                ax.set_title(title, fontsize=10)
                ax.axis("off")
            examples.append(dict(source=source, example=name, image_id=tile.image_id,
                                 tile_id=int(tile.tile_id), overlap_px=int(tile.overlap_px),
                                 region_px=int(tile.region_px), overlap_fraction=float(tile.overlap_fraction)))
        fig.tight_layout()
        fig.savefig(directory / f"{source}.png", dpi=110, bbox_inches="tight")
        plt.close(fig)
    return examples

In [ ]:
def prepare(cfg):
    root = Path(cfg["output_dir"])
    root.mkdir(parents=True, exist_ok=True)
    dest = root / "prepared"

    # 현재 네 폴더에서 목록을 다시 만들어 변경된 이미지·마스크의 이전 결과 사용을 방지합니다.
    records = scan_visa(Path(cfg["paths"]["VisA"]))
    assign_splits(records, cfg["split_fractions"], cfg["seed"])
    originals = pd.DataFrame([item.row() for item in records])
    tiles = pd.DataFrame(enumerate_tiles(records, cfg))
    checks = audit(originals, tiles)
    counts = population(originals, tiles)
    with FileLock(str(root / "prepared.lock")), tempfile.TemporaryDirectory(dir=root) as tmp:
        stage = Path(tmp) / "prepared"
        stage.mkdir()
        originals.to_csv(stage / "originals.csv", index=False)
        tiles.to_csv(stage / "tiles.csv", index=False)
        hashes = {name: sha256(stage / name) for name in ("originals.csv", "tiles.csv")}
        if dest.exists():
            _, _, _, previous = load_prepared(cfg)
            if hashes != previous["manifest_sha256"]:
                raise ValueError("데이터가 저장된 전처리 결과와 다릅니다. 새 OUTPUT_DIR을 지정하세요.")
            return previous

        write_rows(stage / "counts.csv", counts)
        examples = draw_previews(stage / "previews", originals, tiles, cfg)
        write_rows(stage / "preview_examples.csv", examples)
        report = dict(settings=preparation_settings(cfg), paths=cfg["paths"], counts=counts,
                      leakage=checks, omitted={}, manifest_sha256=hashes,
                      group_counts=originals.groupby("source").group_id.nunique().to_dict(),
                      groups_missing_split=[row for row in counts if not row["groups"]])
        save_json(stage / "summary.json", report)
        stage.rename(dest)
    return report

### 기준 실험과 대조

아래 셀은 새로 만든 원본·타일 표의 주요 열을 기존 `07` 전처리 기록과 비교합니다. 원본은 파일 내용과 분할, 타일은 좌표·라벨·결함 픽셀 수까지 대조합니다. 비교에 필요한 기준 해시는 노트북에 포함되어 있습니다. 기준 데이터의 원본 수는 학습 3,091장, 검증 663장, 시험 662장입니다.

In [ ]:
def compare_reference(directory):
    references = {
        "originals.csv": (
            ["image_id", "board", "group_id", "sha256", "split", "label"],
            "e6ba84d7b34018d982204e99deb6070895d50db213a54334fb6c0f4a5e63bc00",
        ),
        "tiles.csv": (
            ["tile_id", "image_id", "split", "x0", "y0", "x1", "y1", "label", "overlap_px", "region_px"],
            "52018dcea56b412550534c5aaa726986cc8ae56b0cadbec49a4721e02b5aa535",
        ),
    }
    matches = {}
    for name, (columns, expected) in references.items():
        digest = hashlib.sha256()
        with (directory / name).open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                values = [row[column] for column in columns]
                digest.update(json.dumps(values, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
                digest.update(b"\n")
        matches[name] = digest.hexdigest() == expected
    return matches

In [ ]:
prepared_report = prepare(cfg)
prepared_dir = output_dir / "prepared"
counts = pd.DataFrame(prepared_report["counts"]).set_index("split")
display(counts[["originals", "original_normal", "original_defect", "all_tiles",
                "tile_normal", "tile_defect", "tile_excluded"]])
print("원본·타일 분할 누수 검사 통과")
for filename, matches in compare_reference(prepared_dir).items():
    print(f"{filename}  ·  기존 07 전처리와 {'일치' if matches else '다름'}")

preview = prepared_dir / "previews" / "VisA.png"
if preview.is_file():
    display(NotebookImage(filename=str(preview), width=850))

## 4. 학습 입력 구성

RGB 원본에서 타일을 먼저 자른 뒤 양선형 보간으로 224×224픽셀로 줄입니다. 기본 설정은 타일을 `OUTPUT_DIR/cache`에 저장해 에포크마다 디코딩하는 비용을 줄입니다. 기준 데이터에서 캐시는 약 3.7 GiB입니다. `cache_inputs=False`면 같은 자르기·리사이즈를 읽을 때마다 수행합니다.

In [ ]:
def rgb_tile(image, box, size):
    patch = image.crop(box).resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(patch, dtype=np.uint8)


def input_cache(cfg, originals, tiles, report):
    if not cfg["training"]["cache_inputs"]:
        return None
    size = cfg["training"]["image_size"]
    signature = report["manifest_sha256"]
    folder = Path(cfg["output_dir"]) / "cache"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{signature['originals.csv'][:16]}_{signature['tiles.csv'][:16]}_{size}.npy"
    with FileLock(str(path) + ".lock"):
        shape = (len(tiles), size, size, 3)
        if path.is_file():
            existing = np.load(path, mmap_mode="r")
            if existing.shape != shape or existing.dtype != np.uint8:
                raise ValueError(f"Tile cache has wrong shape/dtype: {path}")
            return path
        print(f"Caching {len(tiles):,} cropped RGB tiles ({math.prod(shape) / 2**30:.1f} GiB) in {folder}",
              flush=True)
        paths = {r.image_id: (Path(cfg["paths"][r.source]) / r.path, r.sha256)
                 for r in originals.itertuples()}
        geometry = tiles[["tile_id", *COORDS]].to_numpy(dtype=np.int64)
        with tempfile.TemporaryDirectory(dir=folder) as temp:
            stage = Path(temp) / "inputs.npy"
            cache = np.lib.format.open_memmap(stage, mode="w+", dtype=np.uint8, shape=shape)

            def fill(group):
                image_id, indices = group
                image_path, expected = paths[image_id]
                if sha256(image_path) != expected:
                    raise ValueError(f"Original changed after preparation: {image_path}")
                with Image.open(image_path) as raw:
                    image = raw.convert("RGB")
                for i, x, y, r, b in geometry[indices]:
                    cache[i] = rgb_tile(image, (x, y, r, b), size)

            try:
                with ThreadPoolExecutor(max_workers=4) as pool:
                    for _ in pool.map(fill, tiles.groupby("image_id", sort=False).indices.items()):
                        pass
                cache.flush()
            finally:
                del cache
            stage.replace(path)
    return path

In [ ]:
class TileDataset(Dataset):
    def __init__(self, rows, originals, cfg, cache_path):
        self.indices = rows.tile_id.to_numpy(dtype=np.int64)
        self.labels = rows.label.to_numpy(dtype=np.int64)
        self.cache_path = cache_path
        self.cache = None
        self.size = cfg["training"]["image_size"]
        self.last_path = None
        self.last_image = None
        if cache_path is None:
            paths = {r.image_id: Path(cfg["paths"][r.source]) / r.path for r in originals.itertuples()}
            self.paths = rows.image_id.map(paths).tolist()
            self.boxes = rows[COORDS].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        if self.cache_path is None:
            path = self.paths[i]
            if path != self.last_path:
                with Image.open(path) as raw:
                    self.last_image = raw.convert("RGB")
                self.last_path = path
            image = rgb_tile(self.last_image, tuple(self.boxes[i]), self.size)
        else:
            if self.cache is None:
                self.cache = np.load(self.cache_path, mmap_mode="r")
            image = self.cache[self.indices[i]]
        return image, self.labels[i]


def collate(batch):
    images, labels = zip(*batch)
    return torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2), torch.tensor(labels, dtype=torch.long)


def loader(dataset, cfg, device, *, shuffle=False, sampler=None):
    tr = cfg["training"]
    return DataLoader(dataset, batch_size=tr["batch_size"], shuffle=shuffle, sampler=sampler,
                      num_workers=tr["workers"], pin_memory=device.type == "cuda", collate_fn=collate,
                      persistent_workers=tr["workers"] > 0,
                      generator=torch.Generator().manual_seed(cfg["seed"]))

## 5. EfficientNet-B0와 결함 점수

ImageNet 사전학습 모델의 마지막 선형층을 정상·불량 두 출력으로 바꿉니다. 학습 중에는 각 타일에 상하·좌우 반전을 각각 확률 0.5로 적용하고, 모든 입력에 ImageNet 평균과 표준편차를 적용합니다. 검증과 시험은 원본의 모든 타일을 추론한 뒤 가장 높은 불량 확률을 원본 점수로 사용합니다.

In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def device_for(name):
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Set training.device to auto or cpu.")
    return torch.device(name)


def network(name, pretrained):
    if name != "efficientnet_b0":
        raise ValueError(f"이 실험의 모델은 efficientnet_b0입니다. 현재 설정: {name}")
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    return model


def input_batch(images, device, settings, augment=False):
    x = images.to(device, dtype=torch.float32, non_blocking=True).div_(255)
    if settings["preprocessing"] == "binary":
        gray = x[:, 0] * 0.299 + x[:, 1] * 0.587 + x[:, 2] * 0.114
        x = (gray >= settings["binary_cutoff"] / 255.0).float().unsqueeze(1).expand(-1, 3, -1, -1)
    if augment:
        for axis in (2, 3):
            flip = torch.rand((len(x), 1, 1, 1), device=device) < 0.5
            x = torch.where(flip, x.flip((axis,)), x)
    mean = x.new_tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = x.new_tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return ((x - mean) / std).contiguous(memory_format=torch.channels_last)

In [ ]:
@torch.inference_mode()
def batch_scores(model, images, device, settings):
    # Float32 inference is shared by validation, final evaluation and single-image prediction.
    logits = model(input_batch(images, device, settings))
    return logits.float().softmax(dim=1)[:, 1].cpu().numpy().astype(np.float64)


def score_loader(model, batches, device, settings):
    model.eval()
    return np.concatenate([batch_scores(model, images, device, settings) for images, _ in batches])


def restore(checkpoint, device):
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model = network(saved["training"]["model"], False)
    model.load_state_dict(saved["state_dict"])
    return model.to(device, memory_format=torch.channels_last).eval(), saved


def image_scores(originals, tiles, probabilities):
    # The input is the complete grid, including partial-overlap tiles excluded from supervision.
    if set(tiles.image_id) != set(originals.image_id) or len(probabilities) != len(tiles):
        raise ValueError("Inference must cover every original and its complete tile grid")
    scored = tiles[["image_id", "tile_id"]].assign(prob_defect=np.asarray(probabilities, dtype=np.float64))
    grouped = scored.groupby("image_id", sort=False)
    maxima = scored.loc[grouped.prob_defect.idxmax()].rename(columns={"tile_id": "max_tile_id"})
    result = originals[["image_id", "source", "board", "group_id", "path", "label"]].merge(
        maxima, on="image_id", validate="one_to_one")
    result["n_tiles"] = result.image_id.map(grouped.size())
    result["prob_normal"] = 1.0 - result.prob_defect
    return result

### 지표와 검증 임계값

양성은 불량입니다. 정확도·정밀도·재현율·특이도·F1·오탐률(FPR)·미탐률(FNR)은 혼동행렬에서 계산합니다. ROC-AUC와 PR-AUC는 원본 코드와 같은 사다리꼴 적분을 사용합니다.

운영 임계값은 검증 원본에서 재현율 90% 이상, 오탐률 5% 이하를 목표로 정합니다. 두 조건을 동시에 만족하기 어려우면 원본의 우선순위에 따라 오탐률 상한 안에서 재현율이 가장 큰 값을 선택합니다. 구체적인 선택 근거와 검증 결과는 `threshold.json`에 저장합니다.

In [ ]:
def ratio(a, b):
    return float(a / b) if b else None


def arrays(labels, scores):
    y = np.asarray(labels)
    p = np.asarray(scores, dtype=np.float64)
    if y.ndim != 1 or y.shape != p.shape or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected matching arrays of binary targets and scores")
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Nonfinite or out-of-range probability")
    return y.astype(np.int64), p


def threshold_counts(y, p, thresholds):
    order = np.argsort(p, kind="stable")
    positive = np.r_[0, np.cumsum(y[order])]
    cuts = np.searchsorted(p[order], thresholds, side="left")
    tp = positive[-1] - positive[cuts]
    fp = len(y) - cuts - tp
    return tp, fp


def area(x, y):
    return float(np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2))


def metrics(labels, scores, threshold):
    y, p = arrays(labels, scores)
    decision = p >= threshold
    tp = int(np.sum(decision & (y == 1)))
    fp = int(np.sum(decision & (y == 0)))
    positives, negatives = int(y.sum()), int(len(y) - y.sum())
    fn, tn = positives - tp, negatives - fp
    roc_auc = pr_auc = None
    if positives and negatives:
        true_pos, false_pos = threshold_counts(y, p, np.unique(p)[::-1])
        recall = np.r_[0, true_pos / positives]
        roc_auc = area(np.r_[0, false_pos / negatives], recall)
        pr_auc = area(recall, np.r_[1, true_pos / (true_pos + false_pos)])
    measures = dict(accuracy=ratio(tp + tn, len(y)), precision=ratio(tp, tp + fp),
                    recall=ratio(tp, positives), specificity=ratio(tn, negatives),
                    f1=ratio(2 * tp, 2 * tp + fp + fn), roc_auc=roc_auc, pr_auc=pr_auc,
                    fpr=ratio(fp, negatives), fnr=ratio(fn, positives))
    undefined = [k for k, v in measures.items() if v is None]
    return dict(n=len(y), n_normal=negatives, n_defect=positives, tn=tn, fp=fp, fn=fn, tp=tp,
                **measures, undefined=", ".join(undefined))


def log_loss(labels, scores):
    y, p = arrays(labels, scores)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log1p(-p)))

In [ ]:
def choose_threshold(validation, policy):
    y, p = arrays(validation.label, validation.prob_defect)
    thresholds = np.unique(np.r_[p, 0.0, 0.5, 1.0])
    tp, fp = threshold_counts(y, p, thresholds)
    positives, negatives = y.sum(), len(y) - y.sum()
    recall = tp / positives if positives else np.full(len(tp), np.nan)
    fpr = fp / negatives if negatives else np.full(len(fp), np.nan)
    curve = pd.DataFrame(dict(threshold=thresholds, recall=recall, fpr=fpr, tp=tp, fp=fp))
    if not positives or not negatives:
        selected = 0.5
        reason = "Validation lacks one or both classes; the default threshold is retained."
        feasible = None
    else:
        cap = fpr <= policy["max_fpr"]
        valid = cap & (recall >= policy["min_recall"])
        feasible = bool(valid.any())
        if feasible:
            candidates = valid
            reason = "Both targets met. Maximize recall, then minimize FPR, then choose closest to 0.5."
        elif cap.any():
            candidates = cap
            reason = "Recall target infeasible under the FPR cap. Maximize recall within the cap."
        else:
            candidates = fpr == fpr.min()
            reason = "FPR cap infeasible. Minimize FPR, then maximize recall."
        options = curve[candidates].assign(distance=lambda frame: abs(frame.threshold - 0.5))
        best = options.sort_values(["recall", "fpr", "distance", "threshold"],
                                   ascending=[False, True, True, True]).iloc[0]
        selected = float(best.threshold)
    result = dict(threshold=selected, selected_on="validation originals", policy=policy,
                  targets_met=feasible, reason=reason, validation=metrics(y, p, selected))
    return result, curve

In [ ]:
def evaluations(predictions, operating):
    scopes = [("all", predictions)]
    scopes.extend((source, predictions[predictions.source == source]) for source in SOURCES)
    scopes.extend((f"VisA/{board}", predictions[(predictions.source == "VisA") &
                                                (predictions.board == board)]) for board in BOARDS)
    rows = []
    for mode, threshold in (("default", 0.5), ("operating", operating)):
        for scope, frame in scopes:
            rows.append(dict(scope=scope, mode=mode, threshold=threshold,
                             **metrics(frame.label, frame.prob_defect, threshold)))
    return pd.DataFrame(rows)


def save_predictions(directory, predictions, operating):
    predictions = predictions.assign(pred_default=(predictions.prob_defect >= 0.5).astype(int),
                                     pred_operating=(predictions.prob_defect >= operating).astype(int))
    predictions.to_csv(directory / "predictions.csv", index=False)
    evaluations(predictions, operating).to_csv(directory / "metrics.csv", index=False)

## 6. 학습과 최적 모델 저장

학습 타일 빈도의 역수에 비례하는 클래스 가중치로 교차 엔트로피 손실을 계산합니다. AdamW의 학습률은 0.0002, 가중치 감쇠는 0.0001입니다. CUDA에서 학습할 때는 혼합 정밀도를 사용하고, 검증·시험 추론은 float32를 사용합니다.

에포크마다 **검증 원본 이미지 로그 손실**을 계산합니다. 손실이 가장 낮은 첫 에포크를 저장하고, 마지막 개선 이후 7에포크 동안 개선이 없으면 종료합니다. 검증 원본의 예측으로 운영 임계값을 정한 뒤 `runs/VisA/best.pt`에 가중치와 함께 저장합니다.

In [ ]:
def protocol_id(cfg, report):
    conditions = dict(manifests=report["manifest_sha256"], settings=report["settings"],
                      training=cfg["training"], threshold=cfg["threshold"],
                      pooling="maximum", checkpoint="min validation image log loss")
    return hashlib.sha256(json.dumps(conditions, sort_keys=True).encode()).hexdigest()


def train_epoch(model, batches, device, settings, optimizer, scaler, weights):
    model.train()
    loss_total = weight_total = 0.0
    targets, probabilities = [], []
    for images, labels in batches:
        images = input_batch(images, device, settings, augment=settings["augment"])
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=scaler.is_enabled()):
            logits = model(images)
            summed = F.cross_entropy(logits.float(), labels, weight=weights, reduction="sum")
            denominator = weights[labels].sum() if weights is not None else len(labels)
            loss = summed / denominator
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_total += summed.detach().item()
        weight_total += float(denominator)
        targets.append(labels.cpu().numpy())
        probabilities.append(logits.detach().float().softmax(1)[:, 1].cpu().numpy())
    return loss_total / weight_total, metrics(np.concatenate(targets), np.concatenate(probabilities), 0.5)

In [ ]:
def save_curves(directory, frame):
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(9, 3.4))
    for field, label in [("train_loss", "Train tiles (objective)"), ("val_tile_loss", "Validation tiles"),
                         ("val_image_loss", "Validation originals")]:
        ax.plot(frame.epoch, frame[field], marker="o", label=label)
    ax.set(title="Cross-entropy by epoch", xlabel="Epoch", ylabel="Loss")
    for field in ("precision", "recall", "f1", "fpr"):
        bx.plot(frame.epoch, frame["val_" + field], marker="o", label=field.upper())
    bx.set(title="Validation originals at 0.5", xlabel="Epoch", ylabel="Fraction", ylim=(-0.03, 1.03))
    for panel in (ax, bx):
        panel.set_xticks(frame.epoch if len(frame) <= 12 else frame.epoch.iloc[::len(frame)//8])
        panel.legend(fontsize=8)
        panel.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(directory / "learning_curves.png", dpi=130)
    plt.close(fig)

In [ ]:
def fit(directory, experiment, cfg, originals, tiles, report, cache):
    tr = cfg["training"]
    device = device_for(tr["device"])
    seed_all(cfg["seed"])
    train_images = originals[(originals.split == "train") & originals.source.isin(EXPERIMENTS[experiment])]
    train_tiles = tiles[(tiles.split == "train") & tiles.image_id.isin(train_images.image_id) & (tiles.label >= 0)]
    val_images = originals[originals.split == "validation"]
    val_tiles = tiles[tiles.split == "validation"]
    counts = np.bincount(train_tiles.label.to_numpy(), minlength=2)
    if not counts.all() or val_images.empty:
        raise ValueError(f"Need both training classes and validation images. Training counts: {counts}")
    weights = None
    sampler = None
    balanced = len(train_tiles) / (2 * counts)
    if tr["imbalance"] == "class_weight":
        weights = torch.tensor(balanced, dtype=torch.float32, device=device)
    elif tr["imbalance"] == "weighted_sampler":
        sampler = WeightedRandomSampler(balanced[train_tiles.label.to_numpy()], len(train_tiles),
                                        replacement=True, generator=torch.Generator().manual_seed(cfg["seed"]))
    train_batches = loader(TileDataset(train_tiles, originals, cfg, cache), cfg, device,
                           shuffle=sampler is None, sampler=sampler)
    val_batches = loader(TileDataset(val_tiles, originals, cfg, cache), cfg, device)
    model = network(tr["model"], tr["pretrained"]).to(device, memory_format=torch.channels_last)
    opt_args = dict(lr=tr["learning_rate"], weight_decay=tr["weight_decay"])
    optimizer = (torch.optim.AdamW(model.parameters(), **opt_args) if tr["optimizer"] == "adamw"
                 else torch.optim.SGD(model.parameters(), momentum=0.9, **opt_args))
    scaler = torch.amp.GradScaler("cuda", enabled=tr["amp"] and device.type == "cuda")
    print(f"{experiment} on {device}: train tiles normal={counts[0]}, defect={counts[1]}; "
          f"validation originals={len(val_images)}", flush=True)
    history = []
    best_loss = float("inf")
    stale_epochs = 0
    patience = tr["early_stopping_patience"]
    usable_val = val_tiles.label.to_numpy() >= 0
    for epoch in range(1, tr["epochs"] + 1):
        if stale_epochs >= patience:
            print(f"VisA early stopping at epoch {epoch - 1}: validation image loss "
                  f"did not improve for {patience} epochs", flush=True)
            break
        loss, training = train_epoch(model, train_batches, device, tr, optimizer, scaler, weights)
        probabilities = score_loader(model, val_batches, device, tr)
        pred = image_scores(val_images, val_tiles, probabilities)
        image_loss = log_loss(pred.label, pred.prob_defect)
        val_tile_loss = (log_loss(val_tiles.label[usable_val], probabilities[usable_val])
                         if usable_val.any() else None)
        validation = metrics(pred.label, pred.prob_defect, 0.5)
        row = dict(epoch=epoch, train_loss=loss, val_tile_loss=val_tile_loss, val_image_loss=image_loss)
        for key in ("accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "fpr", "fnr"):
            row["train_" + key] = training[key]
            row["val_" + key] = validation[key]
        history.append(row)
        pd.DataFrame(history).to_csv(directory / "history.csv", index=False)
        print(f"{experiment} epoch {epoch}/{tr['epochs']} train_loss={loss:.4f} "
              f"val_image_loss={image_loss:.4f} val_recall={validation['recall']}", flush=True)
        if image_loss < best_loss:
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_loss, best_epoch, best_predictions = image_loss, epoch, pred
            stale_epochs = 0
        else:
            stale_epochs += 1
    decision, curve = choose_threshold(best_predictions, cfg["threshold"])
    validation_dir = directory / "validation"
    validation_dir.mkdir()
    save_predictions(validation_dir, best_predictions, decision["threshold"])
    curve.to_csv(validation_dir / "threshold_curve.csv", index=False)
    save_json(directory / "threshold.json", decision)
    save_json(directory / "classes.json", CLASSES)
    save_json(directory / "config.json", cfg)
    split_list = originals[["image_id", "source", "board", "group_id", "label", "split"]].copy()
    split_list["in_training"] = split_list.image_id.isin(train_images.image_id)
    split_list.to_csv(directory / "splits.csv", index=False)
    save_curves(directory, pd.DataFrame(history))
    identity = protocol_id(cfg, report)
    torch.save(dict(state_dict=best_state, training=tr, tiling=cfg["tiling"], classes=CLASSES,
                    pooling="max", threshold=decision["threshold"], calibration=decision,
                    best_epoch=best_epoch, experiment=experiment, protocol_id=identity), directory / "best.pt")
    save_json(directory / "run.json", dict(experiment=experiment, protocol_id=identity,
              manifest_sha256=report["manifest_sha256"], weights_sha256=sha256(directory / "best.pt"),
              seed=cfg["seed"], train_sources=list(EXPERIMENTS[experiment]),
              validation_sources=sorted(val_images.source.unique()), validation_originals=len(val_images),
              train_tiles={"normal": int(counts[0]), "defect": int(counts[1])},
              imbalance=tr["imbalance"], inverse_frequency_weights=balanced.tolist(),
              best_epoch=best_epoch, best_validation_image_log_loss=best_loss,
              checkpoint_rule="minimum validation original-image log loss; first epoch on a tie",
              operating_threshold=decision["threshold"], device=str(device), torch_version=str(torch.__version__)))

In [ ]:
def train(cfg, experiments):
    _, originals, tiles, report = load_prepared(cfg)
    parent = Path(cfg["output_dir"]) / "runs"
    parent.mkdir(exist_ok=True)
    missing = {s for exp in experiments for s in EXPERIMENTS[exp]} - set(originals.source)
    if missing:
        raise ValueError(f"Requested experiments need these missing sources: {sorted(missing)}")
    cache = input_cache(cfg, originals, tiles, report)
    for exp in experiments:
        dest = parent / exp
        with FileLock(str(dest) + ".lock"):
            if dest.exists():
                info = read_json(dest / "run.json")
                if info["protocol_id"] != protocol_id(cfg, report) or sha256(dest / "best.pt") != info["weights_sha256"]:
                    raise ValueError(f"Different run or changed weights at {dest}. Choose a new output_dir.")
                print(f"Already trained: {dest}")
                continue
            with tempfile.TemporaryDirectory(dir=parent) as tmp:
                directory = Path(tmp) / exp
                directory.mkdir()
                fit(directory, exp, cfg, originals, tiles, report, cache)
                directory.rename(dest)

### 학습 실행

완료되면 선택한 에포크와 임계값, 학습 곡선을 표시합니다. 에포크별 수치는 `history.csv`, 검증 원본의 예측과 임계값 후보는 `validation/`에서 확인할 수 있습니다.

In [ ]:
train(cfg, ["VisA"])

run_info = read_json(run_dir / "run.json")
decision = read_json(run_dir / "threshold.json")
history = pd.read_csv(run_dir / "history.csv")
print(f"학습 {len(history)}에포크  ·  선택한 모델 {run_info['best_epoch']}에포크")
print(f"검증 원본 로그 손실  {run_info['best_validation_image_log_loss']:.6f}")
print(f"검증에서 선택한 임계값  {decision['threshold']:.10g}")
display(NotebookImage(filename=str(run_dir / "learning_curves.png"), width=880))

## 7. 시험 원본 성능 평가

`best.pt`를 다시 읽어 시험 원본의 모든 타일을 평가합니다. 표는 기본 임계값 0.5(`default`)와 검증에서 선택한 임계값(`operating`)의 결과를 전체 및 보드별로 보여줍니다. 혼동행렬은 정상·불량 원본의 장수와 각 실제 클래스 안에서의 비율을 함께 표시합니다.

시험 이미지별 결함 확률과 두 판정은 `runs/VisA/test/predictions.csv`, 전체 성능 수치는 `runs/VisA/test/metrics.csv`에 저장합니다.

In [ ]:
def evaluate(cfg, experiments):
    _, originals, tiles, report = load_prepared(cfg)
    parents = originals[originals.split == "test"]
    grid = tiles[tiles.split == "test"]
    ids = "\n".join(sorted(parents.image_id)).encode()
    test_key = hashlib.sha256(ids).hexdigest()
    identity = protocol_id(cfg, report)
    device = device_for(cfg["training"]["device"])
    batches = None
    for experiment in experiments:
        run = Path(cfg["output_dir"]) / "runs" / experiment
        if not (run / "run.json").is_file():
            raise FileNotFoundError(f"위 학습 셀에서 {experiment} 모델을 먼저 학습하세요.")
        with FileLock(str(run / "test.lock")):
            info = read_json(run / "run.json")
            weights = run / "best.pt"
            if info["protocol_id"] != identity or sha256(weights) != info["weights_sha256"]:
                raise ValueError(f"Model or protocol differs from the sealed run: {run}")
            dest = run / "test"
            if dest.exists():
                done = read_json(dest / "evaluation.json")
                if (done["test_ids_sha256"] != test_key or done["weights_sha256"] != info["weights_sha256"]
                        or any(sha256(dest / name) != value for name, value in done["files"].items())):
                    raise ValueError(f"Saved evaluation does not match: {dest}")
                print(f"{experiment}: using saved final test results in {dest}")
                continue
            model, checkpoint = restore(weights, device)
            if checkpoint["protocol_id"] != identity:
                raise ValueError("Checkpoint was trained with a different manifest or protocol")
            if batches is None:
                cache = input_cache(cfg, originals, tiles, report)
                batches = loader(TileDataset(grid, originals, cfg, cache), cfg, device)
            probs = score_loader(model, batches, device, checkpoint["training"]) if len(grid) else np.empty(0)
            pred = image_scores(parents, grid, probs)
            with tempfile.TemporaryDirectory(dir=run) as tmp:
                directory = Path(tmp) / "test"
                directory.mkdir()
                save_predictions(directory, pred, checkpoint["threshold"])
                save_json(directory / "evaluation.json", dict(protocol_id=identity, test_ids_sha256=test_key,
                          weights_sha256=info["weights_sha256"], originals=len(parents), all_tiles=len(grid),
                          operating_threshold=checkpoint["threshold"], threshold_source="validation",
                          files={name: sha256(directory / name) for name in ("predictions.csv", "metrics.csv")}))
                directory.rename(dest)
            print(f"{experiment}: evaluated {len(pred)} test originals ({len(grid)} tiles), saved to {dest}",
                  flush=True)

In [ ]:
def show_confusion_matrices(overall):
    from io import BytesIO

    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.7), constrained_layout=True)
    for ax, mode, title in zip(axes, ("default", "operating"), ("Default", "Validation-selected")):
        row = overall.loc[mode]
        counts = np.array([[row.tn, row.fp], [row.fn, row.tp]], dtype=int)
        totals = counts.sum(axis=1, keepdims=True)
        rates = np.divide(counts, totals, out=np.full(counts.shape, np.nan), where=totals > 0)
        ax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        for (i, j), count in np.ndenumerate(counts):
            rate = rates[i, j]
            label = f"{count:,}\n{rate:.1%}" if np.isfinite(rate) else f"{count:,}\nN/A"
            ax.text(j, i, label, ha="center", va="center", fontsize=13,
                    color="white" if rate > 0.55 else "#14283b")
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Normal", "Defect"],
               yticklabels=["Normal", "Defect"], xlabel="Predicted class", ylabel="Actual class",
               title=f"{title} · threshold {row.threshold:.6g}")
    fig.suptitle("Test originals · count and share within actual class")
    with BytesIO() as image:
        fig.savefig(image, format="png", dpi=120, bbox_inches="tight")
        plt.close(fig)
        display(NotebookImage(data=image.getvalue(), width=880))

In [ ]:
evaluate(cfg, ["VisA"])

test_dir = run_dir / "test"
test_metrics = pd.read_csv(test_dir / "metrics.csv")
scopes = ["all", *(f"VisA/{board}" for board in BOARDS)]
scores = test_metrics.loc[test_metrics.scope.isin(scopes)].set_index(["mode", "scope"])
overall = scores.xs("all", level="scope")
print(f"시험 원본 {int(overall.loc['operating', 'n']):,}장  ·  "
      f"검증에서 고정한 임계값 {overall.loc['operating', 'threshold']:.10g}")
display(scores[["n", "accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]].round(4))
display(scores[["n_normal", "n_defect", "tn", "fp", "fn", "tp", "fpr", "fnr"]].round(4))
show_confusion_matrices(overall)
print(f"최적 모델    {run_dir / 'best.pt'}")
print(f"원본별 예측  {test_dir / 'predictions.csv'}")
print(f"시험 성능표  {test_dir / 'metrics.csv'}")